In [1]:
from pathlib import Path
import ast

PROJECT_ROOT = Path(r"D:\Projects\speech-ai-learning")

print("项目目录存在:", PROJECT_ROOT.exists())

def show_functions(file_path, target_names=None):
    source = file_path.read_text(encoding="utf-8")
    tree = ast.parse(source)

    for node in tree.body:
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            if target_names is None or node.name in target_names:
                print(f"\n===== {node.name} =====")
                print(ast.get_source_segment(source, node))

        # 同时检查模块级变量，例如正则表达式
        elif isinstance(node, ast.Assign):
            names = [
                target.id for target in node.targets
                if isinstance(target, ast.Name)
            ]
            if "_TOKEN_PATTERN" in names:
                print("\n===== _TOKEN_PATTERN =====")
                print(ast.get_source_segment(source, node))


# 1. 检查 normalize.py
normalize_path = PROJECT_ROOT / "src" / "normalize.py"

print("\n========== normalize.py ==========")

show_functions(
    normalize_path,
    {"normalize_score", "mixed_tokens"}
)

# 2. 检查 metrics.py
metrics_path = PROJECT_ROOT / "src" / "metrics.py"

print("\n========== metrics.py ==========")

show_functions(metrics_path)

# 3. 查看现有测试文件
tests_path = PROJECT_ROOT / "tests"

print("\n========== Existing Tests ==========")

if tests_path.exists():
    for path in sorted(tests_path.rglob("test_*.py")):
        print(path.relative_to(PROJECT_ROOT))
else:
    print("tests 目录不存在")

项目目录存在: True

========== normalize.py ==========

===== _TOKEN_PATTERN =====
_TOKEN_PATTERN = re.compile(
    r"[a-z]+(?:[a-z0-9]|-[a-z0-9]+|\.\d+|'[a-z]+)*(?:\+\+|#)?"
    r"|\d+(?:\.\d+)*"
    r"|[\u3400-\u4dbf\u4e00-\u9fff\U00020000-\U0002EBEF]"
    r"|[^\s]"
)

===== normalize_score =====
def normalize_score(text: str) -> str:
    """
    将原始文本转换为统一评分文本。

    规则：
    1. NFKC
    2. 英文小写
    3. 普通标点转为空格
    4. 非数字间的句点转为空格
    5. 折叠空白
    6. 保留未知有效符号
    """
    if not isinstance(text, str):
        raise TypeError("text must be a string")

    #1. Unicode兼容规范化
    text = unicodedata.normalize("NFKC", text)

    #2.英文小写(lower也会处理其他有大小写的文字)
    text = text.lower()

    #对不可安全解释的控制字符显式报错
    for char in text:
        category = unicodedata.category(char)

        if (
            category in {"Cc", "Cf", "Cs"}
            and not char.isspace()
        ):
            raise ValueError(
                f"Unsupported control character: U+{ord(char):04X}"
            )

    #3.普通标点替换为空格
  

In [2]:
from pathlib import Path

PROJECT_ROOT = Path(r"D:\Projects\speech-ai-learning")

fixture_dir = PROJECT_ROOT / "tests" / "fixtures"
fixture_dir.mkdir(parents=True, exist_ok=True)

In [3]:
fixture_path = fixture_dir / "terms_fixture.csv"

fixture_content = '''canonical_id,canonical,aliases
001,beam,[]
002,beam search,[]
003,search algorithm,[]
004,attention,"[""attn""]"
005,transform,[]
'''

fixture_path.write_text(
    fixture_content,
    encoding="utf-8"
)

print("创建成功:", fixture_path)

创建成功: D:\Projects\speech-ai-learning\tests\fixtures\terms_fixture.csv


## 测试

In [12]:
from pathlib import Path
import sys

root = Path(r"D:\Projects\speech-ai-learning")

if root not in sys.path:
    sys.path.insert(0, str(root))

from src.term_eval import load_terms, atomicize
from src.normalize import mixed_tokens

fixture_path = (
    PROJECT_ROOT
    / "tests"
    / "fixtures"
    / "terms_fixture.csv"
)

terms = load_terms(fixture_path)

print("术语形式总数:", len(terms["entries"]))

print("\n词表索引:")
for key, value in terms["matches_by_first_token"].items():
    print(key, [
        (x["canonical_id"], x["tokens"])
        for x in value
    ])

术语形式总数: 6

词表索引:
beam [('002', ('beam', 'search')), ('001', ('beam',))]
search [('003', ('search', 'algorithm'))]
attention [('004', ('attention',))]
attn [('004', ('attn',))]
transform [('005', ('transform',))]


In [6]:
# Test 1：普通术语匹配
x = mixed_tokens("we use attention today")

y = atomicize(x, terms)

print("Test 1:", y)

assert y == [
    "we",
    "use",
    "TERM:004",
    "today"
]


# Test 2：别名匹配
x = mixed_tokens("we use attn today")

y = atomicize(x, terms)

print("Test 2:", y)

assert y == [
    "we",
    "use",
    "TERM:004",
    "today"
]


# Test 3：最长匹配
x = mixed_tokens("beam search algorithm")

y = atomicize(x, terms)

print("Test 3:", y)

assert y == [
    "TERM:002",
    "algorithm"
]


# Test 4：英文整词边界
x = mixed_tokens("transformer")

y = atomicize(x, terms)

print("Test 4:", y)

assert y == ["transformer"]


# Test 5：重复术语
x = mixed_tokens("attention attention today")

y = atomicize(x, terms)

print("Test 5:", y)

assert y == [
    "TERM:004",
    "TERM:004",
    "today"
]


# Test 6：普通 token 保留
x = mixed_tokens("hello world")

y = atomicize(x, terms)

print("Test 6:", y)

assert y == [
    "hello",
    "world"
]


print("\n所有基础测试通过！")

Test 1: ['we', 'use', 'TERM:004', 'today']
Test 2: ['we', 'use', 'TERM:004', 'today']
Test 3: ['TERM:002', 'algorithm']
Test 4: ['transformer']
Test 5: ['TERM:004', 'TERM:004', 'today']
Test 6: ['hello', 'world']

所有基础测试通过！


In [7]:
#补充异常测试
import csv
import tempfile
from pathlib import Path

from src.term_eval import load_terms


def check_load_terms_error(rows, expected_message):

    with tempfile.TemporaryDirectory() as tmp:

        path = Path(tmp) / "terms.csv"

        with path.open(
            "w",
            encoding="utf-8",
            newline=""
        ) as f:

            writer = csv.DictWriter(
                f,
                fieldnames=[
                    "canonical_id",
                    "canonical",
                    "aliases"
                ]
            )

            writer.writeheader()
            writer.writerows(rows)

        try:
            load_terms(path)

        except ValueError as exc:

            assert expected_message in str(exc), str(exc)

            print("PASS:", str(exc))

        else:
            raise AssertionError(
                f"预期抛出 ValueError: {expected_message}"
            )


# Test 7：重复 canonical_id
check_load_terms_error(
    [
        {
            "canonical_id": "001",
            "canonical": "attention",
            "aliases": "[]"
        },
        {
            "canonical_id": "001",
            "canonical": "conformer",
            "aliases": "[]"
        }
    ],
    "重复 canonical_id"
)


# Test 8：不同术语共享别名
check_load_terms_error(
    [
        {
            "canonical_id": "001",
            "canonical": "attention",
            "aliases": '["attn"]'
        },
        {
            "canonical_id": "002",
            "canonical": "focus",
            "aliases": '["attn"]'
        }
    ],
    "别名冲突"
)


# Test 9：非法 JSON
check_load_terms_error(
    [
        {
            "canonical_id": "001",
            "canonical": "attention",
            "aliases": "not-json"
        }
    ],
    "不是合法 JSON"
)


# Test 10：空别名
check_load_terms_error(
    [
        {
            "canonical_id": "001",
            "canonical": "attention",
            "aliases": '[""]'
        }
    ],
    "存在空别名"
)


print("\n异常测试全部通过！")

PASS: 重复 canonical_id: 001
PASS: 别名冲突: 'attn' 同时映射到 001 和 002
PASS: 第 2 行 aliases 不是合法 JSON
PASS: 第 2 行存在空别名

异常测试全部通过！


## 创建tests/test_term_eval.py

In [8]:
test_file = PROJECT_ROOT / "tests" / "test_term_eval.py"

test_content = r'''
import csv
from pathlib import Path

import pytest

from src.term_eval import load_terms, atomicize
from src.normalize import mixed_tokens


FIXTURE_PATH = (
    Path(__file__).parent
    / "fixtures"
    / "terms_fixture.csv"
)


@pytest.fixture
def terms():
    return load_terms(FIXTURE_PATH)


def test_basic_match(terms):
    tokens = mixed_tokens("we use attention today")

    assert atomicize(tokens, terms) == [
        "we", "use", "TERM:004", "today"
    ]


def test_alias_match(terms):
    tokens = mixed_tokens("we use attn today")

    assert atomicize(tokens, terms) == [
        "we", "use", "TERM:004", "today"
    ]


def test_longest_match(terms):
    tokens = mixed_tokens("beam search algorithm")

    assert atomicize(tokens, terms) == [
        "TERM:002", "algorithm"
    ]


def test_english_word_boundary(terms):
    tokens = mixed_tokens("transformer")

    assert atomicize(tokens, terms) == [
        "transformer"
    ]


def test_repeated_terms(terms):
    tokens = mixed_tokens(
        "attention attention today"
    )

    assert atomicize(tokens, terms) == [
        "TERM:004", "TERM:004", "today"
    ]


def test_ordinary_tokens(terms):
    tokens = mixed_tokens("hello world")

    assert atomicize(tokens, terms) == [
        "hello", "world"
    ]


def write_terms_csv(path, rows, fieldnames=None):
    if fieldnames is None:
        fieldnames = [
            "canonical_id",
            "canonical",
            "aliases"
        ]

    with path.open(
        "w",
        encoding="utf-8",
        newline=""
    ) as f:
        writer = csv.DictWriter(
            f,
            fieldnames=fieldnames
        )

        writer.writeheader()
        writer.writerows(rows)


def test_duplicate_canonical_id(tmp_path):
    path = tmp_path / "terms.csv"

    write_terms_csv(path, [
        {
            "canonical_id": "001",
            "canonical": "attention",
            "aliases": "[]"
        },
        {
            "canonical_id": "001",
            "canonical": "conformer",
            "aliases": "[]"
        }
    ])

    with pytest.raises(ValueError, match="重复 canonical_id"):
        load_terms(path)


def test_alias_conflict(tmp_path):
    path = tmp_path / "terms.csv"

    write_terms_csv(path, [
        {
            "canonical_id": "001",
            "canonical": "attention",
            "aliases": '["attn"]'
        },
        {
            "canonical_id": "002",
            "canonical": "focus",
            "aliases": '["attn"]'
        }
    ])

    with pytest.raises(ValueError, match="别名冲突"):
        load_terms(path)


def test_invalid_alias_json(tmp_path):
    path = tmp_path / "terms.csv"

    write_terms_csv(path, [
        {
            "canonical_id": "001",
            "canonical": "attention",
            "aliases": "not-json"
        }
    ])

    with pytest.raises(ValueError, match="不是合法 JSON"):
        load_terms(path)


def test_empty_alias(tmp_path):
    path = tmp_path / "terms.csv"

    write_terms_csv(path, [
        {
            "canonical_id": "001",
            "canonical": "attention",
            "aliases": '[""]'
        }
    ])

    with pytest.raises(ValueError, match="存在空别名"):
        load_terms(path)


def test_missing_required_columns(tmp_path):
    path = tmp_path / "terms.csv"

    write_terms_csv(
        path,
        [{"canonical_id": "001"}],
        fieldnames=["canonical_id"]
    )

    with pytest.raises(ValueError, match="缺少必要字段"):
        load_terms(path)
'''

test_file.write_text(
    test_content,
    encoding="utf-8"
)

print("测试文件创建成功:", test_file)

测试文件创建成功: D:\Projects\speech-ai-learning\tests\test_term_eval.py


## 做自动化测试

In [9]:
import subprocess
import sys

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "tests/test_term_eval.py",
        "-v"
    ],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)

print("Return code:", result.returncode)

============================= test session starts =============================
platform win32 -- Python 3.11.17, pytest-9.1.1, pluggy-1.6.0 -- d:\anaconda3\envs\speech-ai\python.exe
cachedir: .pytest_cache
rootdir: D:\Projects\speech-ai-learning
plugins: anyio-4.15.1, platformdirs-4.12.3
collecting ... collected 11 items

tests/test_term_eval.py::test_basic_match PASSED                         [  9%]
tests/test_term_eval.py::test_alias_match PASSED                         [ 18%]
tests/test_term_eval.py::test_longest_match PASSED                       [ 27%]
tests/test_term_eval.py::test_english_word_boundary PASSED               [ 36%]
tests/test_term_eval.py::test_repeated_terms PASSED                      [ 45%]
tests/test_term_eval.py::test_ordinary_tokens PASSED                     [ 54%]
tests/test_term_eval.py::test_duplicate_canonical_id PASSED              [ 63%]
tests/test_term_eval.py::test_alias_conflict PASSED                      [ 72%]
tests/test_term_eval.py::test_inval

## 测试Term-WER计算

In [19]:
from pathlib import Path
import sys

root = Path(r"D:\Projects\speech-ai-learning")

if root not in sys.path:
    sys.path.insert(0, str(root))

import importlib
import src.term_eval
importlib.reload(src.term_eval)

from src.term_eval import (
    align_sequences,
    compute_term_stats
)

import math

def run_term_test(
    name,
    reference,
    hypothesis,
    expected
):
    # 1. 完整句子对齐
    alignment = align_sequences(
        reference,
        hypothesis
    )

    # 2. 术语错误统计
    result = compute_term_stats(
        reference,
        alignment["operations"]
    )

    # 3. 检查预期结果
    for key, expected_value in expected.items():

        actual_value = result[key]

        if expected_value is None:

            assert actual_value is None, (
                f"{name}: {key} 预期为 None，"
                f"实际为 {actual_value}"
            )

        elif isinstance(expected_value, float):

            assert math.isclose(
                actual_value,
                expected_value,
                rel_tol=1e-9
            ), (
                f"{name}: {key} 预期为 {expected_value}，"
                f"实际为 {actual_value}"
            )

        else:

            assert actual_value == expected_value, (
                f"{name}: {key} 预期为 {expected_value}，"
                f"实际为 {actual_value}"
            )

    print(f"\n===== {name} PASSED =====")
    print("Alignment:", alignment["operations"])
    print("Term stats:", result)

    return result

In [20]:
run_term_test(
    "Test 1: Term Substitution",

    reference=[
        "TERM:001"
    ],

    hypothesis=[
        "TERM:002"
    ],

    expected={
        "S_t": 1,
        "D_t": 0,
        "I_t": 0,
        "N_t": 1,
        "term_wer": 1.0
    }
)

run_term_test(
    "Test 2: Normal Token to Term",

    reference=[
        "study"
    ],

    hypothesis=[
        "TERM:002"
    ],

    expected={
        "S_t": 0,
        "D_t": 0,
        "I_t": 1,
        "N_t": 0,
        "term_wer": None
    }
)

run_term_test(
    "Test 3: Term Deletion",

    reference=[
        "TERM:001"
    ],

    hypothesis=[],

    expected={
        "S_t": 0,
        "D_t": 1,
        "I_t": 0,
        "N_t": 1,
        "term_wer": 1.0
    }
)

run_term_test(
    "Test 4: Term Insertion",

    reference=[],

    hypothesis=[
        "TERM:001"
    ],

    expected={
        "S_t": 0,
        "D_t": 0,
        "I_t": 1,
        "N_t": 0,
        "term_wer": None
    }
)

run_term_test(
    "Test 5: Repeated Terms",

    reference=[
        "TERM:001",
        "TERM:001"
    ],

    hypothesis=[
        "TERM:001"
    ],

    expected={
        "S_t": 0,
        "D_t": 1,
        "I_t": 0,
        "N_t": 2,
        "term_wer": 0.5
    }
)

run_term_test(
    "Test 6: Normal Token Substitution",

    reference=[
        "today"
    ],

    hypothesis=[
        "tomorrow"
    ],

    expected={
        "S_t": 0,
        "D_t": 0,
        "I_t": 0,
        "N_t": 0,
        "term_wer": None
    }
)

run_term_test(
    "Test 7: Zero Reference Terms with Two Insertions",

    reference=[
        "study",
        "today"
    ],

    hypothesis=[
        "TERM:001",
        "TERM:002"
    ],

    expected={
        "S_t": 0,
        "D_t": 0,
        "I_t": 2,
        "N_t": 0,
        "term_wer": None
    }
)

run_term_test(
    "Test 8: Combined Errors",

    reference=[
        "we",
        "TERM:001",
        "study",
        "TERM:004"
    ],

    hypothesis=[
        "we",
        "TERM:002",
        "TERM:003"
    ],

    expected={
        "S_t": 1,
        "D_t": 1,
        "I_t": 1,
        "N_t": 2,
        "term_wer": 1.5
    }
)


===== Test 1: Term Substitution PASSED =====
Alignment: [('S', 'TERM:001', 'TERM:002')]
Term stats: {'S_t': 1, 'D_t': 0, 'I_t': 0, 'N_t': 1, 'term_wer': 1.0}

===== Test 2: Normal Token to Term PASSED =====
Alignment: [('S', 'study', 'TERM:002')]
Term stats: {'S_t': 0, 'D_t': 0, 'I_t': 1, 'N_t': 0, 'term_wer': None}

===== Test 3: Term Deletion PASSED =====
Alignment: [('D', 'TERM:001', None)]
Term stats: {'S_t': 0, 'D_t': 1, 'I_t': 0, 'N_t': 1, 'term_wer': 1.0}

===== Test 4: Term Insertion PASSED =====
Alignment: [('I', None, 'TERM:001')]
Term stats: {'S_t': 0, 'D_t': 0, 'I_t': 1, 'N_t': 0, 'term_wer': None}

===== Test 5: Repeated Terms PASSED =====
Alignment: [('D', 'TERM:001', None), ('C', 'TERM:001', 'TERM:001')]
Term stats: {'S_t': 0, 'D_t': 1, 'I_t': 0, 'N_t': 2, 'term_wer': 0.5}

===== Test 6: Normal Token Substitution PASSED =====
Alignment: [('S', 'today', 'tomorrow')]
Term stats: {'S_t': 0, 'D_t': 0, 'I_t': 0, 'N_t': 0, 'term_wer': None}

===== Test 7: Zero Reference Terms

{'S_t': 1, 'D_t': 1, 'I_t': 1, 'N_t': 2, 'term_wer': 1.5}

## 测试语料级统计

In [21]:
from pathlib import Path
import sys

root = Path(r"D:\Projects\speech-ai-learning")

if root not in sys.path:
    sys.path.insert(0, str(root))

import importlib
import src.term_eval
importlib.reload(src.term_eval)

from src.term_eval import score_term_corpus

import math

In [22]:
sample_a = (
    ["TERM:001"],
    ["TERM:002"]
)

sample_b = (
    ["TERM:001"] * 9,
    ["TERM:001"] * 9
)

result = score_term_corpus([
    sample_a,
    sample_b
])

print(result)

assert result["S_t"] == 1
assert result["D_t"] == 0
assert result["I_t"] == 0
assert result["N_t"] == 10
assert result["term_errors"] == 1
assert result["term_wer"] == 0.1

print("Test 1 PASS")

result = score_term_corpus([
    (
        [],
        ["TERM:001", "TERM:002"]
    )
])

print(result)

assert result["S_t"] == 0
assert result["D_t"] == 0
assert result["I_t"] == 2
assert result["N_t"] == 0
assert result["term_wer"] is None

print("Test 2 PASS")

result = score_term_corpus([
    (
        ["TERM:001"],
        ["TERM:001"]
    ),
    (
        [],
        ["TERM:002"]
    )
])

print(result)

assert result["S_t"] == 0
assert result["D_t"] == 0
assert result["I_t"] == 1
assert result["N_t"] == 1
assert result["term_wer"] == 1.0

print("Test 3 PASS")

result = score_term_corpus([
    (
        ["TERM:001", "TERM:001"],
        ["TERM:001"]
    ),
    (
        ["TERM:002", "TERM:002"],
        ["TERM:002", "TERM:002"]
    )
])

print(result)

assert result["S_t"] == 0
assert result["D_t"] == 1
assert result["I_t"] == 0
assert result["N_t"] == 4
assert result["term_wer"] == 0.25

print("Test 4 PASS")

result = score_term_corpus([])

print(result)

assert result["S_t"] == 0
assert result["D_t"] == 0
assert result["I_t"] == 0
assert result["N_t"] == 0
assert result["term_errors"] == 0
assert result["term_wer"] is None
assert result["sample_results"] == []

print("Test 5 PASS")

{'S_t': 1, 'D_t': 0, 'I_t': 0, 'N_t': 10, 'term_errors': 1, 'term_wer': 0.1, 'sample_results': [{'sample_index': 1, 'S_t': 1, 'D_t': 0, 'I_t': 0, 'N_t': 1, 'term_wer': 1.0}, {'sample_index': 2, 'S_t': 0, 'D_t': 0, 'I_t': 0, 'N_t': 9, 'term_wer': 0.0}]}
Test 1 PASS
{'S_t': 0, 'D_t': 0, 'I_t': 2, 'N_t': 0, 'term_errors': 2, 'term_wer': None, 'sample_results': [{'sample_index': 1, 'S_t': 0, 'D_t': 0, 'I_t': 2, 'N_t': 0, 'term_wer': None}]}
Test 2 PASS
{'S_t': 0, 'D_t': 0, 'I_t': 1, 'N_t': 1, 'term_errors': 1, 'term_wer': 1.0, 'sample_results': [{'sample_index': 1, 'S_t': 0, 'D_t': 0, 'I_t': 0, 'N_t': 1, 'term_wer': 0.0}, {'sample_index': 2, 'S_t': 0, 'D_t': 0, 'I_t': 1, 'N_t': 0, 'term_wer': None}]}
Test 3 PASS
{'S_t': 0, 'D_t': 1, 'I_t': 0, 'N_t': 4, 'term_errors': 1, 'term_wer': 0.25, 'sample_results': [{'sample_index': 1, 'S_t': 0, 'D_t': 1, 'I_t': 0, 'N_t': 2, 'term_wer': 0.5}, {'sample_index': 2, 'S_t': 0, 'D_t': 0, 'I_t': 0, 'N_t': 2, 'term_wer': 0.0}]}
Test 4 PASS
{'S_t': 0, 'D_t':